# Course 5: Building a Small Bangla Language Model
## Session 3: Building the corpus

**Goal:** build a cleaned, deduplicated corpus with a document-level split and a data card.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every code cell here was run once on a CPU (November 2026) using the small built-in sample. Cells marked TODO or NOT RUN need your data, a GPU or the network. The download cell was not run.

In [ ]:
# On Colab or Kaggle, run once:  !pip install datasets regex

In [ ]:
SAMPLE_LINES = [
    "আমি বাংলায় গান গাই।",
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "বাংলা আমার মাতৃভাষা।",
    "আজ আকাশ খুব পরিষ্কার।",
    "ছোট মেয়েটি নদীর ধারে খেলছিল।",
    "বাবা বাজার থেকে মাছ কিনে আনলেন।",
    "মা রান্নাঘরে ভাত রান্না করছেন।",
    "আমরা সবাই মিলে স্কুলে যাই।",
    "বৃষ্টি হলে মাঠে পানি জমে।",
    "পাখিরা সকালে গান গায়।",
    "দাদু প্রতিদিন সন্ধ্যায় গল্প বলেন।",
    "ছেলেটি বই পড়তে খুব ভালোবাসে।",
    "নদীতে অনেক নৌকা চলছে।",
    "শীতকালে আমরা পিঠা খাই।",
    "ঢাকা বাংলাদেশের রাজধানী।",
    "আমার বন্ধু ভালো ফুটবল খেলে।",
]

### 1. Load documents
The sample below stands in for real documents. The commented cell loads Bangla Wikipedia; check the dataset page for current configuration names.

In [ ]:
# NOT RUN: needs the network.
# from datasets import load_dataset
# ds = load_dataset("wikimedia/wikipedia", "20231101.bn", split="train")
# docs = [row["text"] for row in ds]

In [ ]:
import hashlib, random, unicodedata
import regex
# stand-in documents with some noise and repeats
docs = [" ".join(SAMPLE_LINES[i:i + 4]) + "\n\nআরও পড়ুন\n\nHome | About | Contact" for i in range(0, 16, 2)] * 3
print(len(docs), "documents")

### 2. Clean, filter and deduplicate, keeping counts

In [ ]:
def clean(text):
    text = unicodedata.normalize("NFC", text).replace("\u200b", "")
    return regex.sub(r"[ \t\u00a0]+", " ", text).strip()

def bangla_ratio(text):
    letters = regex.findall(r"\p{L}", text)
    return sum(1 for c in letters if "\u0980" <= c <= "\u09ff") / len(letters) if letters else 0.0

MIN_CHARS, MIN_RATIO = 20, 0.8
seen, stats, kept_docs = set(), {"paragraphs": 0, "short": 0, "non_bangla": 0, "duplicate": 0, "kept": 0}, []
for doc in docs:
    kept = []
    for p in doc.split("\n\n"):
        p = clean(p)
        stats["paragraphs"] += 1
        if len(p) < MIN_CHARS:
            stats["short"] += 1; continue
        if bangla_ratio(p) < MIN_RATIO:
            stats["non_bangla"] += 1; continue
        h = hashlib.md5(p.encode("utf-8")).hexdigest()
        if h in seen:
            stats["duplicate"] += 1; continue
        seen.add(h); kept.append(p); stats["kept"] += 1
    if kept:
        kept_docs.append(kept)
print(stats)

### 3. Split by document, after deduplication

In [ ]:
random.seed(0)
random.shuffle(kept_docs)
n = len(kept_docs)
n_val = max(1, n // 100); n_test = max(1, n // 100)
splits = {"val": kept_docs[:n_val], "test": kept_docs[n_val:n_val + n_test], "train": kept_docs[n_val + n_test:]}
for name, part in splits.items():
    text = "\n".join("\n".join(d) for d in part)
    with open(f"{name}.txt", "w", encoding="utf-8") as f:
        f.write(text)
    print(name, len(part), "docs,", len(text), "characters")

### 4. Read your data
Print random paragraphs and note every kind of problem you see.

In [ ]:
all_paragraphs = [p for d in splits["train"] for p in d]
for p in random.sample(all_paragraphs, min(5, len(all_paragraphs))):
    print("-", p[:120])

### 5. Data card (TODO)
- Sources and licenses:
- Cleaning steps and thresholds:
- Sizes before and after each step:
- Known gaps:
- Personal-data risk: